# TACO Dataset Waste Classification

This notebook trains a state-of-the-art image classification model for waste classification using the TACO dataset.

**Key Features:**
- Pre-cropped images for maximum GPU utilization (90%+ vs 10-20%)
- ConvNeXt-Large architecture with advanced regularization
- CrossEntropyLoss with Class Weights for handling class imbalance
- CLASS_MAP categorization: plastic, glass, paper, unsorted
- Comprehensive F1 score tracking (macro, weighted, per-class)
- GPU acceleration with optimized data loading (8 workers, batch size 32)


Responsible for this notebook and task: Bumin Cetin


## 1. Setup and Imports


In [ ]:
import os
import json
import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models, datasets
from tqdm import tqdm
import random
from collections import defaultdict, Counter
from sklearn.metrics import f1_score, precision_recall_fscore_support

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_MAP = {
    # plastic
    "clear plastic bottle": "plastic",
    "crisp packet": "plastic",
    "disposable food container": "plastic",
    "disposable plastic cup": "plastic",
    "foam cup": "plastic",
    "foam food container": "plastic",
    "garbage bag": "plastic",
    "other plastic": "plastic",
    "other plastic bottle": "plastic",
    "other plastic container": "plastic",
    "other plastic cup": "plastic",
    "other plastic wrapper": "plastic",
    "plastic bottle cap": "plastic",
    "plastic film": "plastic",
    "plastic glooves": "plastic",
    "plastic lid": "plastic",
    "plastic straw": "plastic",
    "plastic utensils": "plastic",
    "polypropylene bag": "plastic",
    "single-use carrier bag": "plastic",
    "six pack rings": "plastic",
    "spread tub": "plastic",
    "squeezable tube": "plastic",
    "styrofoam piece": "plastic",
    "tupperware": "plastic",

    # glass
    "broken glass": "glass",
    "glass bottle": "glass",
    "glass cup": "glass",
    "glass jar": "glass",

    # paper
    "corrugated carton": "paper",
    "drink carton": "paper",
    "egg carton": "paper",
    "magazine paper": "paper",
    "meal carton": "paper",
    "normal paper": "paper",
    "other carton": "paper",
    "paper bag": "paper",
    "paper cup": "paper",
    "paper straw": "paper",
    "pizza box": "paper",
    "tissues": "paper",
    "toilet tube": "paper",
    "wrapping paper": "paper",

    # unsorted (metal, food, mixed, unknown)
    "aerosol": "unsorted",
    "aluminium blister pack": "unsorted",
    "aluminium foil": "unsorted",
    "battery": "unsorted",
    "carded blister pack": "unsorted",
    "cigarette": "unsorted",
    "drink can": "unsorted",
    "food can": "unsorted",
    "food waste": "unsorted",
    "metal bottle cap": "unsorted",
    "metal lid": "unsorted",
    "plastified paper bag": "unsorted",  # composite
    "pop tab": "unsorted",
    "rope & strings": "unsorted",
    "scrap metal": "unsorted",
    "shoe": "unsorted",
    "unlabeled litter": "unsorted",
}

def get_class_from_category(cat_obj):
    """
    Map TACO category name to new classification category using CLASS_MAP.

    Args:
        cat_obj: Category object from TACO annotations (has 'name' field)

    Returns:
        str: Mapped category name ('plastic', 'glass', 'paper', or 'unsorted')
    """
    if not cat_obj:
        return "unsorted"

    category_name = cat_obj.get("name", "").lower().strip()

    # Direct lookup in CLASS_MAP
    if category_name in CLASS_MAP:
        return CLASS_MAP[category_name]

    return "unsorted"


Using device: cuda
GPU: Tesla T4
CUDA Version: 12.6

✓ CLASS_MAP loaded with 60 category mappings
✓ Categories: ['glass', 'paper', 'plastic', 'unsorted']


## 0. Pre-Process Images (Pre-Crop for Speed)

**IMPORTANT**: Run this cell ONCE to pre-crop all images. This eliminates CPU bottleneck and allows GPU to run at full capacity.

After running this, you can use ImageFolder dataset which is much faster than on-the-fly cropping.

**NOTE**: This notebook now uses CLASS_MAP to categorize images into 4 classes: `plastic`, `glass`, `paper`, and `unsorted`. If you have existing pre-cropped images created with the old supercategory system, you should delete the `PROCESSED_DIR` directory and re-run this cell to recreate images with the new CLASS_MAP categories.


### Mount Google Drive

Mounting Google Drive is necessary to persist the downloaded dataset or if the dataset is already stored there.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Download TACO Dataset

We will now run the `data_import.ipynb` notebook, which will download the TACO dataset and place it in the expected directory (`./TACO`). This is crucial for the rest of the notebook to function correctly.

In [3]:
# Run the data_import.ipynb notebook to download the TACO dataset
%run data_import.ipynb

✅ TACO repository cloned successfully!

✅ Dependencies installed successfully!
Installing COCO API...
✅ COCO API installed successfully!

Checking if dataset needs to be downloaded...
Data directory exists but images not found. Starting download...
This may take a while depending on your internet connection...
✅ Download script completed!
Note. If for any reason the connection is broken. Just call me again and I will start where I left.
Loading: [..............................] - 0/1500
Loading: [..............................] - 1/1500
Loading: [..............................] - 2/1500
Loading: [..............................] - 3/1500
Loading: [..............................] - 4/1500
Loading: [..............................] - 5/1500
Loading: [..............................] - 6/1500
Loading: [..............................] - 7/1500
Loading: [..............................] - 8/1500
Loading: [..............................] - 9/1500
Loading: [..............................] - 10/15

In [ ]:
import os
import json
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
import random
from collections import defaultdict
import shutil

ROOT = "./TACO"
ANN_PATH = os.path.join(ROOT, "data", "annotations.json")
IMG_ROOT = os.path.join(ROOT, "data")
PROCESSED_DIR = "./taco_cropped_v1"
MIN_CROP_SIZE = 32
VAL_SPLIT = 0.15
TEST_SPLIT = 0.1

# Check if pre-cropped images already exist and use CLASS_MAP
def check_existing_cropped_images(processed_dir):
    """Check if cropped images already exist and use CLASS_MAP categories."""
    if not os.path.exists(processed_dir):
        return False

    # Get expected CLASS_MAP classes
    if 'CLASS_MAP' not in globals():
        return False  # Can't verify without CLASS_MAP

    expected_classes = sorted(set(CLASS_MAP.values()))  # ['glass', 'paper', 'plastic', 'unsorted']

    # Check if train, val, test directories exist
    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(processed_dir, split)
        if not os.path.exists(split_dir):
            return False

        # Get class directories
        class_dirs = sorted([d for d in os.listdir(split_dir)
                           if os.path.isdir(os.path.join(split_dir, d)) and not d.startswith('.')])

        if not class_dirs:
            return False

        # Verify classes match CLASS_MAP (at least for train split)
        if split == 'train':
            if set(class_dirs) != set(expected_classes):
                return False  # Train must have all CLASS_MAP categories

        # Check if at least one class has images
        has_images = False
        for class_dir in class_dirs:
            class_path = os.path.join(split_dir, class_dir)
            image_files = [f for f in os.listdir(class_path)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
            if len(image_files) > 0:
                has_images = True
                break

        if not has_images:
            return False

    return True

if check_existing_cropped_images(PROCESSED_DIR):
    pass
else:
    if os.path.exists(PROCESSED_DIR):
        if 'CLASS_MAP' not in globals():
            raise RuntimeError("CLASS_MAP not found! Please run the Setup and Imports cell first.")
        shutil.rmtree(PROCESSED_DIR)

    if 'CLASS_MAP' not in globals():
        raise RuntimeError("CLASS_MAP not found! Please run the Setup and Imports cell first.")
    if 'get_class_from_category' not in globals():
        raise RuntimeError("get_class_from_category function not found! Please run the Setup and Imports cell first.")

    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(PROCESSED_DIR, split), exist_ok=True)

    try:
        with open(ANN_PATH, 'r', encoding='utf-8') as f:
            ann_data = json.load(f)
    except FileNotFoundError:
        raise FileNotFoundError(f"Annotations file not found at: {ANN_PATH}")
    except json.JSONDecodeError as e:
        raise ValueError(f"Error parsing annotations JSON file: {e}")

    id2img = {im["id"]: im for im in ann_data["images"]}
    id2cat = {c["id"]: c for c in ann_data["categories"]}

    def safe_crop(img, x, y, w, h):
        H, W = img.shape[:2]
        x0, y0 = max(0, int(x)), max(0, int(y))
        x1, y1 = min(W, int(x + w)), min(H, int(y + h))
        if x1 <= x0 or y1 <= y0:
            return None
        return img[y0:y1, x0:x1]

    def mask_crop(img, ann_obj, use_mask=True):
        seg = ann_obj.get("segmentation", [])
        if use_mask and isinstance(seg, list) and len(seg) > 0 and isinstance(seg[0], list):
            mask = np.zeros(img.shape[:2], dtype=np.uint8)
            for poly in seg:
                pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
                cv2.fillPoly(mask, [pts.astype(np.int32)], 255)
            ys, xs = np.where(mask > 0)
            if len(xs) > 0 and len(ys) > 0:
                x0, x1 = xs.min(), xs.max()
                y0, y1 = ys.min(), ys.max()
                crop = img[y0:y1+1, x0:x1+1]
                mask_crop = mask[y0:y1+1, x0:x1+1]
                crop = cv2.bitwise_and(crop, crop, mask=mask_crop)
                return crop
        x, y, w, h = ann_obj["bbox"]
        return safe_crop(img, x, y, w, h)

    class_names = sorted(set(CLASS_MAP.values()))

    for split in ['train', 'val', 'test']:
        for class_name in class_names:
            os.makedirs(os.path.join(PROCESSED_DIR, split, class_name), exist_ok=True)

    annotations = ann_data["annotations"].copy()
    random.seed(42)
    random.shuffle(annotations)

    total = len(annotations)
    num_test = int(total * TEST_SPLIT)
    num_val = int(total * VAL_SPLIT)
    num_train = total - num_val - num_test

    train_ann = annotations[:num_train]
    val_ann = annotations[num_train:num_train + num_val]
    test_ann = annotations[num_train + num_val:]

    splits = {
        'train': train_ann,
        'val': val_ann,
        'test': test_ann
    }

    stats = defaultdict(int)

    for split_name, ann_list in splits.items():
        skipped_count = 0
        for ann_idx, ann in enumerate(tqdm(ann_list, desc=f"Cropping {split_name}")):
            img_info = id2img.get(ann["image_id"])
            if not img_info:
                skipped_count += 1
                continue

            img_path = os.path.join(IMG_ROOT, img_info["file_name"])
            if not os.path.exists(img_path):
                skipped_count += 1
                continue

            try:
                img = cv2.imread(img_path)
                if img is None:
                    skipped_count += 1
                    continue
            except Exception:
                skipped_count += 1
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            try:
                crop = mask_crop(img, ann, use_mask=True)
                if crop is None or crop.shape[0] < MIN_CROP_SIZE or crop.shape[1] < MIN_CROP_SIZE:
                    skipped_count += 1
                    continue
            except Exception:
                skipped_count += 1
                continue

            cat = id2cat.get(ann["category_id"])
            if not cat:
                continue

            label = get_class_from_category(cat)

            if label not in class_names:
                continue

            try:
                crop_pil = Image.fromarray(crop)
                save_path = os.path.join(PROCESSED_DIR, split_name, label, f"{ann['image_id']}_{ann['id']}.jpg")
                crop_pil.save(save_path, quality=95)
                stats[f"{split_name}_{label}"] += 1
            except Exception:
                skipped_count += 1
                continue


✓ Found TACO dataset at: ./TACO
✓ Annotations path: ./TACO/data/annotations.json
✓ Image root: ./TACO/data
✓ Output directory: ./taco_cropped_v1
Pre-cropped images not found or incomplete in: ./taco_cropped_v1
Starting preprocessing...
Loading annotations...
✓ Loaded 1500 images and 60 categories from annotations

CLASS_MAP Validation:
  Total mappings: 60
  Target classes: ['glass', 'paper', 'plastic', 'unsorted']

  Mapping breakdown:
    glass: 4 TACO categories
    paper: 14 TACO categories
    plastic: 25 TACO categories
    unsorted: 17 TACO categories
Using CLASS_MAP: Found 4 classes: ['glass', 'paper', 'plastic', 'unsorted']

Splits: Train=3589, Val=717, Test=478

Processing TRAIN split (3589 annotations)...


Cropping train: 100%|██████████| 3589/3589 [06:24<00:00,  9.32it/s]


  Note: Skipped 486 annotations (missing images, invalid crops, etc.)

Processing VAL split (717 annotations)...


Cropping val: 100%|██████████| 717/717 [01:14<00:00,  9.66it/s]


  Note: Skipped 95 annotations (missing images, invalid crops, etc.)

Processing TEST split (478 annotations)...


Cropping test: 100%|██████████| 478/478 [00:48<00:00,  9.82it/s]

  Note: Skipped 49 annotations (missing images, invalid crops, etc.)

Pre-processing complete!

Statistics (using CLASS_MAP categories):

TRAIN: 3103 images
  glass: 169 (5.4%)
  paper: 382 (12.3%)
  plastic: 1592 (51.3%)
  unsorted: 960 (30.9%)

VAL: 622 images
  glass: 23 (3.7%)
  paper: 69 (11.1%)
  plastic: 325 (52.3%)
  unsorted: 205 (33.0%)

TEST: 429 images
  glass: 17 (4.0%)
  paper: 43 (10.0%)
  plastic: 230 (53.6%)
  unsorted: 139 (32.4%)

✓ All images pre-cropped and saved to: ./taco_cropped_v1
✓ Images organized by CLASS_MAP categories: ['glass', 'paper', 'plastic', 'unsorted']
✓ You can now use ImageFolder dataset for much faster training!

✓ Verifying directory structure...
  ✓ train: All CLASS_MAP categories present
  ✓ val: All CLASS_MAP categories present
  ✓ test: All CLASS_MAP categories present


## 2. Configuration


In [ ]:
PROCESSED_DIR = "./taco_cropped_v1"

possible_roots = ["./TACO", "../TACO", "TACO"]

ROOT = None
for root_path in possible_roots:
    ann_path = os.path.join(root_path, "data", "annotations.json")
    if os.path.exists(ann_path):
        ROOT = root_path
        break

if ROOT is None:
    raise FileNotFoundError(f"TACO dataset not found! Tried paths: {possible_roots}")

ANN_PATH = os.path.join(ROOT, "data", "annotations.json")
IMG_ROOT = os.path.join(ROOT, "data")

BATCH_SIZE = 32
NUM_EPOCHS = 25
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MIN_CROP_SIZE = 32
VAL_SPLIT = 0.15
TEST_SPLIT = 0.1
GRADIENT_ACCUMULATION_STEPS = 1
NUM_WORKERS = 8
MODEL_SAVE_PATH = "./convnext_taco_classification.pth"


## 3. Custom Dataset Class

This dataset loads images and crops objects on-the-fly using ground truth bounding boxes from annotations.


In [ ]:
class TACOClassificationDataset(Dataset):
    def __init__(self, annotations, id2img, id2cat, img_root, transform=None, use_mask=True, cat_name_to_idx=None):
        self.annotations = annotations
        self.id2img = id2img
        self.id2cat = id2cat
        self.img_root = img_root
        self.transform = transform
        self.use_mask = use_mask
        self.cat_name_to_idx = cat_name_to_idx

        self.valid_indices = []
        for idx, ann in enumerate(annotations):
            img_info = id2img.get(ann["image_id"])
            if not img_info:
                continue
            img_path = os.path.join(img_root, img_info["file_name"])
            if not os.path.exists(img_path):
                continue
            self.valid_indices.append(idx)

    def __len__(self):
        return len(self.valid_indices)

    def safe_crop(self, img, x, y, w, h):
        H, W = img.shape[:2]
        x0, y0 = max(0, int(x)), max(0, int(y))
        x1, y1 = min(W, int(x + w)), min(H, int(y + h))
        if x1 <= x0 or y1 <= y0:
            return None
        return img[y0:y1, x0:x1]

    def mask_crop(self, img, ann_obj):
        seg = ann_obj.get("segmentation", [])
        if self.use_mask and isinstance(seg, list) and len(seg) > 0 and isinstance(seg[0], list):
            mask = np.zeros(img.shape[:2], dtype=np.uint8)
            for poly in seg:
                pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
                cv2.fillPoly(mask, [pts.astype(np.int32)], 255)
            ys, xs = np.where(mask > 0)
            if len(xs) > 0 and len(ys) > 0:
                x0, x1 = xs.min(), xs.max()
                y0, y1 = ys.min(), ys.max()
                crop = img[y0:y1+1, x0:x1+1]
                mask_crop = mask[y0:y1+1, x0:x1+1]
                crop = cv2.bitwise_and(crop, crop, mask=mask_crop)
                return crop
        x, y, w, h = ann_obj["bbox"]
        return self.safe_crop(img, x, y, w, h)

    def __getitem__(self, idx):
        ann_idx = self.valid_indices[idx]
        ann = self.annotations[ann_idx]

        img_info = self.id2img[ann["image_id"]]
        img_path = os.path.join(self.img_root, img_info["file_name"])
        img = cv2.imread(img_path)
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        crop = self.mask_crop(img, ann)
        if crop is None or crop.shape[0] < MIN_CROP_SIZE or crop.shape[1] < MIN_CROP_SIZE:
            crop = np.zeros((224, 224, 3), dtype=np.uint8)

        crop_pil = Image.fromarray(crop)

        cat = self.id2cat.get(ann["category_id"])
        label = get_class_from_category(cat)

        if self.transform:
            crop_pil = self.transform(crop_pil)

        return crop_pil, label


## 4. Load Annotations and Prepare Data


In [ ]:
try:
    with open(ANN_PATH, 'r', encoding='utf-8') as f:
        ann_data = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(f"Annotations file not found at: {ANN_PATH}")
except json.JSONDecodeError as e:
    raise ValueError(f"Error parsing annotations JSON file: {e}")

id2img = {im["id"]: im for im in ann_data["images"]}
id2cat = {c["id"]: c for c in ann_data["categories"]}

class_names = sorted(set(CLASS_MAP.values()))
cat_name_to_idx = {name: idx for idx, name in enumerate(class_names)}
num_classes = len(class_names)


Loading annotations...
Total images: 1500
Total annotations: 4784
Total categories: 60

Using CLASS_MAP: Found 4 classes:
  1. glass
  2. paper
  3. plastic
  4. unsorted

Number of classes: 4

CLASS_MAP Statistics:
  glass: 4 TACO categories mapped
  paper: 14 TACO categories mapped
  plastic: 25 TACO categories mapped
  unsorted: 17 TACO categories mapped


## 5. Create Data Transforms

Using advanced augmentation techniques for better generalization.


In [ ]:
class AdaptiveAugmentation:
    def __init__(self, minority_classes, cat_name_to_idx):
        self.minority_classes = set([cat_name_to_idx[name] for name in minority_classes])
        self.strong_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(30),
            transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.2),
            transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), scale=(0.7, 1.3)),
            transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=0.5, scale=(0.02, 0.4))
        ])
        self.normal_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __call__(self, img, label_idx):
        if label_idx in self.minority_classes:
            return self.strong_transform(img)
        return self.normal_transform(img)

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


## 6. Split Data into Train/Val/Test Sets


In [ ]:
annotations = ann_data["annotations"].copy()
random.shuffle(annotations)

total = len(annotations)
num_test = int(total * TEST_SPLIT)
num_val = int(total * VAL_SPLIT)
num_train = total - num_val - num_test

train_ann = annotations[:num_train]
val_ann = annotations[num_train:num_train + num_val]
test_ann = annotations[num_train + num_val:]

temp_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

temp_train_dataset = TACOClassificationDataset(
    train_ann, id2img, id2cat, IMG_ROOT,
    transform=temp_transform, use_mask=True
)


Train samples: 3589
Val samples: 717
Test samples: 478
Total: 4784
Created temporary dataset for class analysis

Data loaders created successfully!


## 6.5. Class Imbalance Analysis and Solutions

TACO dataset has severe class imbalance. Let's calculate class weights and implement solutions.


Class imbalance analysis will run automatically in Cell 17 after datasets are loaded.


In [ ]:
if not os.path.exists(PROCESSED_DIR):
    raise FileNotFoundError(f"Pre-cropped images directory not found: {PROCESSED_DIR}")

expected_classes = sorted(set(CLASS_MAP.values()))
train_root = os.path.join(PROCESSED_DIR, 'train')

if os.path.exists(train_root):
    actual_classes = sorted([d for d in os.listdir(train_root)
                            if os.path.isdir(os.path.join(train_root, d)) and not d.startswith('.')])
    if set(actual_classes) != set(expected_classes):
        raise ValueError(
            f"Pre-cropped images use old class structure. "
            f"Expected {len(expected_classes)} CLASS_MAP classes, found {len(actual_classes)} classes. "
            f"Please delete '{PROCESSED_DIR}' and re-run preprocessing."
        )

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.33))
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def get_all_class_dirs(root_dir):
    if not os.path.exists(root_dir):
        return []
    class_dirs = [d for d in os.listdir(root_dir)
                  if os.path.isdir(os.path.join(root_dir, d)) and not d.startswith('.')]
    return sorted(class_dirs)

def count_images_in_class(root_dir, class_name):
    class_path = os.path.join(root_dir, class_name)
    if not os.path.exists(class_path):
        return 0
    image_files = [f for f in os.listdir(class_path)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    return len(image_files)

val_root = os.path.join(PROCESSED_DIR, 'val')
test_root = os.path.join(PROCESSED_DIR, 'test')

master_classes = expected_classes

train_dataset = datasets.ImageFolder(train_root, transform=train_transform)
val_dataset = datasets.ImageFolder(val_root, transform=val_transform)
test_dataset = datasets.ImageFolder(test_root, transform=val_transform)

train_classes = train_dataset.classes
train_class_to_idx = train_dataset.class_to_idx

def remap_dataset_classes(dataset, target_classes, target_class_to_idx):
    original_classes = dataset.classes
    original_class_to_idx = dataset.class_to_idx
    remapped_samples = []
    remapped_targets = []

    for path, original_target_idx in dataset.samples:
        original_class = original_classes[original_target_idx]
        if original_class in target_class_to_idx:
            new_target_idx = target_class_to_idx[original_class]
            remapped_samples.append((path, new_target_idx))
            remapped_targets.append(new_target_idx)

    dataset.samples = remapped_samples
    dataset.targets = remapped_targets
    dataset.classes = target_classes
    dataset.class_to_idx = target_class_to_idx
    return dataset

val_dataset = remap_dataset_classes(val_dataset, train_classes, train_class_to_idx)
test_dataset = remap_dataset_classes(test_dataset, train_classes, train_class_to_idx)

supercategory_names = train_dataset.classes
num_classes = len(supercategory_names)
cat_name_to_idx = {name: idx for idx, name in enumerate(supercategory_names)}

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False,
    persistent_workers=True if NUM_WORKERS > 0 else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False,
    persistent_workers=True if NUM_WORKERS > 0 else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False,
    persistent_workers=True if NUM_WORKERS > 0 else False
)

from collections import Counter

label_indices = []
for i in range(len(train_dataset)):
    _, label_idx = train_dataset[i]
    label_indices.append(label_idx)

class_counts = Counter(label_indices)
total_samples = len(label_indices)

class_weights = torch.zeros(num_classes)
for class_idx in range(num_classes):
    count = class_counts.get(class_idx, 1)
    class_weights[class_idx] = total_samples / (num_classes * count)


✓ Verified: Pre-cropped images use CLASS_MAP categories: ['glass', 'paper', 'plastic', 'unsorted']
Loading pre-cropped images from: ./taco_cropped_v1

Using CLASS_MAP: Found 4 classes: ['glass', 'paper', 'plastic', 'unsorted']

Class distribution across splits:
----------------------------------------------------------------------
Class                             Train      Val     Test
----------------------------------------------------------------------
glass                               169       23       17
paper                               382       69       43
plastic                            1592      325      230
unsorted                            960      205      139

Loading datasets...
  Train: Loading all classes...
  Val: Loading (empty class directories will be skipped automatically)...
  Test: Loading (empty class directories will be skipped automatically)...

✓ Datasets loaded successfully!
  Classes: 4
  Train: 3103 samples
  Val: 622 samples
  Test: 429 sampl

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(



Class distribution in training set (3103 samples):
------------------------------------------------------------
glass                      0:  169 samples (  5.4%)
paper                      1:  382 samples ( 12.3%)
plastic                    2: 1592 samples ( 51.3%)
unsorted                   3:  960 samples ( 30.9%)
------------------------------------------------------------

Class weights (inverse frequency): tensor([4.5902, 2.0308, 0.4873, 0.8081])
Weight range: 0.49 - 4.59

✓ Class weights calculated and ready for training!
✓ Will use CrossEntropyLoss with class weights


## 7. Create Model - ConvNeXt (State-of-the-Art)

ConvNeXt is a modern CNN architecture that achieves state-of-the-art performance, combining the best of CNNs and Transformers.


In [ ]:
model = models.convnext_large(weights=models.ConvNeXt_Large_Weights.IMAGENET1K_V1)

model.classifier = nn.Sequential(
    nn.Flatten(start_dim=1),
    nn.LayerNorm((1536,), eps=1e-6, elementwise_affine=True),
    nn.Dropout(0.3),
    nn.Linear(1536, 512),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(512, num_classes)
)

model = model.to(device)


Loading ConvNeXt-Large model...
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:04<00:00, 190MB/s]


Model created with 4 output classes
Model parameters: 197,019,332
Trainable parameters: 197,019,332


## 8. Training Setup

Using AdamW optimizer with cosine annealing learning rate scheduler.


In [ ]:
if 'class_weights' not in globals():
    raise RuntimeError("class_weights not found!")

class_weights = class_weights.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=LEARNING_RATE * 0.01
)


✓ Loss: CrossEntropyLoss with inverse frequency class weights
  Weight range: 0.49 - 4.59
✓ Optimizer: AdamW (lr=0.0001, weight_decay=0.0001)
✓ Scheduler: CosineAnnealingLR (T_max=25, eta_min=1.0000000000000002e-06)

✓ Training setup complete! Ready to train.


## 9. Training Loop


In [ ]:
def train_epoch(model, loader, optimizer, criterion, device, accumulation_steps=1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training")
    for batch_idx, (images, labels) in enumerate(pbar):
        labels = labels.to(device)
        images = images.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix({
            'loss': f'{running_loss/(batch_idx+1):.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def validate_with_f1(model, loader, criterion, device):
    model.eval()
    all_preds = []
    all_targets = []
    running_loss = 0.0

    with torch.no_grad():
        pbar = tqdm(loader, desc="Validating")
        for images, labels in pbar:
            labels = labels.to(device)
            images = images.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

            total = len(all_preds)
            correct = sum([p == t for p, t in zip(all_preds[-len(predicted):], labels.cpu().numpy())])
            current_acc = 100. * sum([p == t for p, t in zip(all_preds, all_targets)]) / len(all_preds)
            pbar.set_postfix({
                'loss': f'{running_loss/len(loader):.4f}',
                'acc': f'{current_acc:.2f}%'
            })

    accuracy = 100. * sum([p == t for p, t in zip(all_preds, all_targets)]) / len(all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_targets, all_preds, average='weighted', zero_division=0)

    precision, recall, f1_per_class, support = precision_recall_fscore_support(
        all_targets, all_preds, average=None, zero_division=0
    )

    return running_loss/len(loader), accuracy, macro_f1, weighted_f1, (precision, recall, f1_per_class, support), all_preds, all_targets

def log_detailed_metrics(all_targets, all_preds, supercategory_names, epoch, phase='Val'):
    print(f"\n{'='*80}")
    print(f"Epoch {epoch} - {phase} Per-Class Metrics")
    print(f"{'='*80}")

    all_labels = list(range(len(supercategory_names)))

    precision, recall, f1, support = precision_recall_fscore_support(
        all_targets, all_preds,
        average=None,
        zero_division=0,
        labels=all_labels
    )

    class_metrics = []
    for idx, name in enumerate(supercategory_names):
        class_metrics.append({
            'name': name,
            'f1': f1[idx],
            'precision': precision[idx],
            'recall': recall[idx],
            'support': support[idx]
        })

    class_metrics.sort(key=lambda x: x['f1'])

    print(f"\n{'Class':<30} {'F1':>6} {'Prec':>6} {'Rec':>6} {'Support':>8}")
    print('-' * 80)
    for m in class_metrics:
        print(f"{m['name']:<30} {m['f1']:>6.3f} {m['precision']:>6.3f} {m['recall']:>6.3f} {m['support']:>8}")

    macro_f1 = f1.mean()
    weighted_f1 = (f1 * support).sum() / support.sum() if support.sum() > 0 else 0
    print('-' * 80)
    print(f"{'Macro F1':<30} {macro_f1:>6.3f}")
    print(f"{'Weighted F1':<30} {weighted_f1:>6.3f}")
    print(f"{'='*80}\n")

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

best_macro_f1 = 0.0
train_losses = []
train_accs = []
val_losses = []
val_accs = []
val_macro_f1s = []
val_weighted_f1s = []

for epoch in range(NUM_EPOCHS):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, criterion, device
    )
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    val_loss, val_acc, macro_f1, weighted_f1, class_metrics, all_val_preds, all_val_targets = validate_with_f1(
        model, val_loader, criterion, device
    )
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    val_macro_f1s.append(macro_f1)
    val_weighted_f1s.append(weighted_f1)

    precision, recall, f1_per_class, support = class_metrics

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print(f"Macro F1: {macro_f1:.4f} | Weighted F1: {weighted_f1:.4f}")
    print(f"Learning Rate: {current_lr:.6f}")

    if (epoch + 1) % 5 == 0 or macro_f1 > best_macro_f1:
        log_detailed_metrics(all_val_targets, all_val_preds, supercategory_names, epoch+1, 'Val')

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_macro_f1': macro_f1,
            'val_weighted_f1': weighted_f1,
            'num_classes': num_classes,
            'category_names': supercategory_names,
            'cat_name_to_idx': cat_name_to_idx
        }, MODEL_SAVE_PATH)
        print(f"✓ Saved Best Model (Macro F1: {macro_f1:.4f}, Acc: {val_acc:.2f}%)")


GPU Memory cleared. Free: 0.80 GB
Starting training...
Device: cuda
Number of epochs: 25
Batch size: 32
Gradient accumulation steps: 1
Effective batch size: 32
Learning rate: 0.0001
------------------------------------------------------------

Epoch 1/25
------------------------------------------------------------


Validating: 100%|██████████| 20/20 [00:16<00:00,  1.19it/s, loss=1.1447, acc=50.00%]


Train Loss: 1.0193 | Train Acc: 50.02%
Val Loss: 1.1447 | Val Acc: 50.00%
Macro F1: 0.4708 | Weighted F1: 0.5388
Learning Rate: 0.000100

Epoch 1 - Val Per-Class Metrics

Class                              F1   Prec    Rec  Support
--------------------------------------------------------------------------------
glass                           0.224  0.126  1.000       23
plastic                         0.518  0.820  0.378      325
paper                           0.531  0.435  0.681       69
unsorted                        0.610  0.648  0.576      205
--------------------------------------------------------------------------------
Macro F1                        0.471
Weighted F1                     0.539

✓ Saved Best Model (Macro F1: 0.4708, Acc: 50.00%)
------------------------------------------------------------

Epoch 2/25
------------------------------------------------------------


Validating: 100%|██████████| 20/20 [00:14<00:00,  1.36it/s, loss=0.8455, acc=64.15%]


Train Loss: 0.7057 | Train Acc: 65.78%
Val Loss: 0.8455 | Val Acc: 64.15%
Macro F1: 0.5844 | Weighted F1: 0.6571
Learning Rate: 0.000098

Epoch 2 - Val Per-Class Metrics

Class                              F1   Prec    Rec  Support
--------------------------------------------------------------------------------
glass                           0.480  0.346  0.783       23
paper                           0.490  0.370  0.725       69
unsorted                        0.672  0.702  0.644      205
plastic                         0.696  0.806  0.612      325
--------------------------------------------------------------------------------
Macro F1                        0.584
Weighted F1                     0.657

✓ Saved Best Model (Macro F1: 0.5844, Acc: 64.15%)
------------------------------------------------------------

Epoch 3/25
------------------------------------------------------------


...

Train Loss: 0.0423 | Train Acc: 98.03%
Val Loss: 0.8843 | Val Acc: 78.62%
Macro F1: 0.7331 | Weighted F1: 0.7857
Learning Rate: 0.000003
------------------------------------------------------------

Epoch 24/25
------------------------------------------------------------


Validating: 100%|██████████| 20/20 [00:14<00:00,  1.36it/s, loss=0.8864, acc=78.78%]


Train Loss: 0.0452 | Train Acc: 97.36%
Val Loss: 0.8864 | Val Acc: 78.78%
Macro F1: 0.7336 | Weighted F1: 0.7869
Learning Rate: 0.000001
------------------------------------------------------------

Epoch 25/25
------------------------------------------------------------


Validating: 100%|██████████| 20/20 [00:15<00:00,  1.30it/s, loss=0.8873, acc=78.62%]

Train Loss: 0.0404 | Train Acc: 98.23%
Val Loss: 0.8873 | Val Acc: 78.62%
Macro F1: 0.7310 | Weighted F1: 0.7852
Learning Rate: 0.000001

Epoch 25 - Val Per-Class Metrics

Class                              F1   Prec    Rec  Support
--------------------------------------------------------------------------------
paper                           0.662  0.688  0.638       69
glass                           0.667  0.682  0.652       23
unsorted                        0.760  0.764  0.756      205
plastic                         0.836  0.826  0.846      325
--------------------------------------------------------------------------------
Macro F1                        0.731
Weighted F1                     0.785

------------------------------------------------------------

Training completed!
Best validation accuracy: 78.94%
Best macro F1: 0.7406
Best weighted F1: 0.7885


## 10. Evaluate on Test Set


In [ ]:
checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_acc, test_macro_f1, test_weighted_f1, test_class_metrics, test_preds, test_targets = validate_with_f1(model, test_loader, criterion, device)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Test Macro F1: {test_macro_f1:.4f}")
print(f"Test Weighted F1: {test_weighted_f1:.4f}")

log_detailed_metrics(test_targets, test_preds, supercategory_names, checkpoint['epoch'], 'Test')


Loading best model for testing...
Loaded model from epoch 15 with val acc: 78.94%

Evaluating on test set...


Validating:   0%|          | 0/14 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Validating: 100%|██████████| 14/14 [00:13<00:00,  1.06it/s, loss=0.7856, acc=77.39%]


Test Loss: 0.7856
Test Accuracy: 77.39%
Test Macro F1: 0.7458
Test Weighted F1: 0.7746

Epoch 15 - Test Per-Class Metrics

Class                              F1   Prec    Rec  Support
--------------------------------------------------------------------------------
paper                           0.660  0.608  0.721       43
unsorted                        0.743  0.769  0.719      139
glass                           0.765  0.765  0.765       17
plastic                         0.816  0.814  0.817      230
--------------------------------------------------------------------------------
Macro F1                        0.746
Weighted F1                     0.775



## 11. Per-Class Accuracy Analysis


In [ ]:
model.eval()
class_correct = defaultdict(int)
class_total = defaultdict(int)

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Computing per-class accuracy"):
        labels = labels.to(device)
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)

        for i in range(labels.size(0)):
            label = labels[i].item()
            class_correct[label] += (predicted[i] == label).item()
            class_total[label] += 1

print("\nPer-Supercategory Test Accuracy:")
print("-" * 60)
for idx, class_name in enumerate(supercategory_names):
    if class_total[idx] > 0:
        acc = 100. * class_correct[idx] / class_total[idx]
        print(f"{class_name:30s}: {acc:6.2f}% ({class_correct[idx]}/{class_total[idx]})")
    else:
        print(f"{class_name:30s}: No samples")
print("-" * 60)


Computing per-class accuracy: 100%|██████████| 14/14 [00:10<00:00,  1.33it/s]


Per-Supercategory Test Accuracy (for reference):
------------------------------------------------------------
glass                         :  76.47% (13/17)
paper                         :  72.09% (31/43)
plastic                       :  81.74% (188/230)
unsorted                      :  71.94% (100/139)
------------------------------------------------------------


## 12. Model Summary

The trained model is saved and ready for inference. Key features:

- **Architecture**: ConvNeXt-Large (state-of-the-art CNN)
- **Input**: Images cropped using ground truth bounding boxes
- **Output**: Classification into 4 CLASS_MAP categories: `plastic`, `glass`, `paper`, `unsorted`
- **Model saved at**: `./convnext_taco_classification.pth`

To use the model for inference, load it with:
```python
# Note: Use weights_only=False for trusted checkpoint files (PyTorch 2.6+)
checkpoint = torch.load('convnext_taco_classification.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
class_names = checkpoint['category_names']  # Contains CLASS_MAP categories: ['glass', 'paper', 'plastic', 'unsorted']
```
